# Day 3: Baseline ML — Vietnamese Price Prediction

**Dataset:** `SeanSunny/items_tv_v6` (110K train / 5K val / 5K test)

**Metrics:** RMSLE (primary), MAE (VND), MAPE (%), R2

**Machine:** AMD Ryzen 5 7500F (6C/12T), 28GB RAM, RTX 5060 Ti 16GB

### Toi uu:
- Tokenize **1 lan duy nhat** -> luu pickle cache
- Tat ca models dung chung tokenized data
- `n_jobs=6` (6 cores), `Pool(10)` (12 threads, giu 2 cho OS)
- XGBoost dung GPU (`device='cuda'`)

In [1]:
!nvidia-smi

Tue Apr 14 10:54:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   44C    P8              7W /  180W |     373MiB /  16311MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
#!uv pip uninstall torch torchvision torchaudio

In [3]:
#!uv pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130

In [4]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
GPU: NVIDIA GeForce RTX 5060 Ti


In [5]:
import torch

# Kiểm tra xem CUDA có khả dụng không
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Current Device ID: {torch.cuda.current_device()}")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 5060 Ti
GPU Count: 1
Current Device ID: 0


In [6]:
import random
import time
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from pricer_vi.items import Item
from pricer_vi.evaluator import evaluate

SEED = 42
DATASET = "SeanSunny/items_tv_v6"
N_JOBS = 6         # 6 cores
N_WORKERS = 4     # 12 threads - 2 cho OS

CACHE_DIR = Path("day3")
CACHE_DIR.mkdir(exist_ok=True)
TRAIN_CACHE = CACHE_DIR / "tokenized_train.pkl"
TEST_CACHE = CACHE_DIR / "tokenized_test.pkl"

## 1. Load Data

In [7]:
train, val, test = Item.from_hub(DATASET)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test items")
print(f"\nSample: {test[0]}")
print(f"\nSummary: {test[0].summary[:300]}")

Loaded 110,000 train, 5,000 val, 5,000 test items

Sample: title='Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L HULKER NEW màu Xanh quân đội| Index Living Mall' category='Nhà Cửa - Đời Sống' price=479400 full=None brand=None summary='Tiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.' prompt='Sản phẩm này giá bao nhiêu?\n\nTiêu đề: Thùng lưu trữ đa năng 30L màu xanh quân đội  \nDanh mục: Hộp đựng, Thùng lưu trữ  \nThương hiệu: Index Living Mall  \nMô tả: Hộp đựng đa năng bằng nhựa PP cao cấp, thiết kế hiện đại, phù hợp cho nhiều mục đích lưu trữ.  \nThông số: Dung tích 30L, kích thước 39x39x36cm, màu xanh lá quân đội, chắc chắn, chống thấm nước.\n\nGiá: 479400' id=None

Summary: Tiêu đề: Thùng lưu t

In [8]:
train_prices = [item.price for item in train]
prices = np.array(train_prices, dtype=float)
documents = [item.summary for item in train]

print(f"Price range: {min(train_prices):,} - {max(train_prices):,} VND")
print(f"Mean: {np.mean(train_prices):,.0f} | Median: {np.median(train_prices):,.0f} | Std: {np.std(train_prices):,.0f} VND")

Price range: 4,900 - 50,000,000 VND
Mean: 1,302,725 | Median: 302,000 | Std: 3,583,425 VND


---
## Step 3: Baselines (Random, Mean, Median)

In [9]:
min_price, max_price = min(train_prices), max(train_prices)

def random_pricer(item):
    return random.randint(min_price, max_price)

random.seed(SEED)
np.random.seed(SEED)
results_random = evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

42,436,706 7,377,201 1,657,343 49,409,511 14,481,805 16,313,314 14,299,319 9,085,131 49,087,646 6,833,234 43,467,433 49,562,270 36,334,828 1,249,266 39,387,181 27,792,594 2,362,201 1,952,557 4,225,681 3,302,554 11,118,508 31,658,719 40,081,693 1,596,698 37,539,418 13,199,668 47,946,162 43,394,906 46,636,522 35,625,303 27,935,398 14,633,419 26,190,808 39,449,850 18,374,962 245,024 10,604,455 45,054,241 27,551,072 22,719,725 18,481,530 10,359,952 14,386,361 22,404,377 6,788,275 6,139,968 25,361,389 3,005,426 22,705,598 21,129,377 22,266,268 17,587,594 2,026,809 41,411,371 30,001,381 35,695,558 8,158,341 24,957,913 12,696,909 36,913,719 19,589,762 41,412,202 41,463,057 24,108,815 38,664,346 12,760,168 47,224,665 4,307,667 2,992,122 42,981,530 14,319,894 19,350,397 5,140,648 14,980,673 6,578,741 25,189,739 11,247,568 30,282,750 41,839,960 24,268,373 10,807,431 24,762,324 23,646,713 13,839,678 43,269,594 16,501,478 46,844,263 45,360,542 42,913,818 1,226,641 40,807,989 42,382,571 11,370,820 

In [10]:
training_average = sum(train_prices) / len(train_prices)
print(f"Training average: {training_average:,.0f} VND")

def mean_pricer(item):
    return training_average

results_mean = evaluate(mean_pricer, test)

Training average: 1,302,725 VND


  0%|          | 0/200 [00:00<?, ?it/s]

823,325 1,203,725 1,276,725 942,725 2,677,275 1,176,725 617,725 1,018,725 961,725 1,252,725 647,275 1,154,725 1,033,725 3,287,275 1,057,725 775,725 3,197,275 1,250,725 764,275 16,677,275 3,197,275 957,275 978,725 1,113,725 1,172,725 1,153,725 1,192,725 1,079,725 869,225 352,725 1,079,725 1,137,725 2,657,275 1,202,725 1,003,725 1,106,725 1,187,725 499,275 487,725 1,183,725 1,131,725 1,223,725 1,234,725 1,113,725 1,226,885 1,213,725 1,162,725 2,187,275 87,275 655,275 16,951,275 1,133,725 408,725 6,262,275 467,725 1,007,725 1,079,225 852,726 16,687,275 1,164,725 1,212,726 522,725 1,252,725 1,137,725 1,216,725 1,153,725 1,239,475 937,725 1,214,725 97,275 323,725 1,227,725 1,083,725 656,167 1,098,475 977,725 6,109,099 1,152,725 477,725 1,082,725 1,189,725 1,217,725 1,102,725 1,077,725 407,275 117,275 1,043,725 786,260 722,725 2,267,275 1,227,725 1,067,725 1,183,725 234,605 1,113,725 852,725 1,113,725 1,183,725 752,725 1,103,725 953,725 1,001,725 1,057,725 612,725 1,052,725 612,725 1,028,779

In [11]:
training_median = float(np.median(train_prices))
print(f"Training median: {training_median:,.0f} VND")

def median_pricer(item):
    return training_median

results_median = evaluate(median_pricer, test)

Training median: 302,000 VND


  0%|          | 0/200 [00:00<?, ?it/s]

177,400 203,000 276,000 58,000 3,678,000 176,000 383,000 18,000 39,000 252,000 1,648,000 154,000 33,000 4,288,000 57,000 225,000 4,198,000 250,000 1,765,000 17,678,000 4,198,000 1,958,000 22,000 113,000 172,000 153,000 192,000 79,000 131,500 648,000 79,000 137,000 3,658,000 202,000 3,000 106,000 187,000 1,500,000 513,000 183,000 131,000 223,000 234,000 113,000 226,160 213,000 162,000 3,188,000 1,088,000 1,656,000 17,952,000 133,000 592,000 7,263,000 533,000 7,000 78,500 147,999 17,688,000 164,000 212,001 478,000 252,000 137,000 216,000 153,000 238,750 63,000 214,000 1,098,000 677,000 227,000 83,000 344,558 97,750 23,000 7,109,824 152,000 523,000 82,000 189,000 217,000 102,000 77,000 1,408,000 1,118,000 43,000 214,465 278,000 3,268,000 227,000 67,000 183,000 766,120 113,000 148,000 113,000 183,000 248,000 103,000 47,000 1,000 57,000 388,000 52,000 388,000 2,029,504 3,548,000 4,978,000 253,000 105,700 153,000 203,000 193,000 814,000 203,000 247,000 1,548,000 103,000 15,683,000 149,000 58

---
## Step 4: LR + TF-IDF — Architecture A (n-gram, khong tach tu)

In [12]:
np.random.seed(SEED)
t0 = time.time()
vectorizer_a = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_a = vectorizer_a.fit_transform(documents)
print(f"TF-IDF A: {X_train_a.shape} | Time: {time.time() - t0:.1f}s")
print(f"Sample features: {list(vectorizer_a.get_feature_names_out()[2500:2520])}")

TF-IDF A: (110000, 10000) | Time: 9.1s
Sample features: ['dễ lau', 'dễ làm', 'dễ lắp', 'dễ mang', 'dễ mix', 'dễ pha', 'dễ phối', 'dễ sử', 'dễ thay', 'dễ tháo', 'dễ thương', 'dễ vệ', 'dễ điều', 'dệt', 'dệt kim', 'dị', 'dị ứng', 'dịch', 'dịch thông', 'dịch và']


In [13]:
t0 = time.time()
lr_model_a = LinearRegression()
lr_model_a.fit(X_train_a, prices)
print(f"LR A training: {time.time() - t0:.1f}s")

def lr_tfidf_arch_a(item):
    x = vectorizer_a.transform([item.summary])
    return max(lr_model_a.predict(x)[0], 0)

results_lr_a = evaluate(lr_tfidf_arch_a, test)

LR A training: 12.2s


  0%|          | 0/200 [00:00<?, ?it/s]

462,616 99,000 26,000 1,948,705 9,945,727 126,000 343,700 483,933 19,775 188,420 2,859,994 390,381 257,660 1,450,736 245,000 945,729 1,488,011 52,000 1,844,537 11,619,624 2,645,216 4,239,748 324,000 189,000 37,081 724,083 182,012 823,929 14,940 574,965 49,651 165,000 46,741 92,045 190,964 381,959 115,000 1,093,589 270,889 55,256 798,742 91,317 68,000 118,601 75,840 89,000 4,087 1,263,259 556,689 1,000,403 340,066 169,000 94,290 3,263,147 1,820,269 1,692,047 223,500 337,668 8,040,604 138,000 215,404 780,000 50,000 165,000 131,924 149,000 63,250 3,169,247 495,922 1,254,202 128,982 75,000 1,090,607 646,558 204,250 1,168,698 1,632,791 2,325,136 1,856,917 295,492 113,000 348,195 405,329 225,000 2,877,082 2,554,326 259,000 68,180 209,390 1,244,606 788,560 449,895 119,000 579,158 278,854 1,554,968 1,821,222 36,919 1,071,769 199,000 72,249 1,632,078 245,000 1,156,116 193,067 831,751 305,485 1,630,032 1,786,765 49,000 193,712 47,073 165,189 2,048,966 167,000 99,000 1,334,760 1,322,399 199,000 4

---
## Step 5: Underthesea Tokenize (1 lan duy nhat, luu cache)

Cell nay tokenize **ca train va test**, luu vao `day3/*.pkl`.

Lan dau: ~20-30s (10 workers). Lan sau: load tu file ~1-2s.

In [ ]:
from underthesea import word_tokenize
from multiprocessing import Pool

def tokenize_one(text):
    return word_tokenize(text, format="text")

# --- Tokenize TRAIN (1 lan, luu cache) ---
if TRAIN_CACHE.exists():
    print(f"Loading cached train tokens from {TRAIN_CACHE}...")
    with open(TRAIN_CACHE, "rb") as f:
        tokenized_train = pickle.load(f)
    print(f"Loaded {len(tokenized_train):,} docs (from cache).")
else:
    t0 = time.time()
    print(f"Pre-tokenizing {len(documents):,} train docs ({N_WORKERS} workers)...")
    with Pool(N_WORKERS) as p:
        tokenized_train = p.map(tokenize_one, documents)
    print(f"Tokenization: {time.time() - t0:.1f}s")
    with open(TRAIN_CACHE, "wb") as f:
        pickle.dump(tokenized_train, f)
    print(f"Saved to {TRAIN_CACHE}")

# --- Tokenize TEST (1 lan, luu cache) ---
if TEST_CACHE.exists():
    print(f"Loading cached test tokens from {TEST_CACHE}...")
    with open(TEST_CACHE, "rb") as f:
        tokenized_test = pickle.load(f)
    print(f"Loaded {len(tokenized_test):,} docs (from cache).")
else:
    test_summaries = [item.summary for item in test]
    t0 = time.time()
    print(f"Pre-tokenizing {len(test_summaries):,} test docs ({N_WORKERS} workers)...")
    with Pool(N_WORKERS) as p:
        tokenized_test = p.map(tokenize_one, test_summaries)
    print(f"Tokenization: {time.time() - t0:.1f}s")
    with open(TEST_CACHE, "wb") as f:
        pickle.dump(tokenized_test, f)
    print(f"Saved to {TEST_CACHE}")

# Build lookup map for evaluate()
tokenized_test_map = {item.summary: tok for item, tok in zip(test, tokenized_test)}

print(f"\nOriginal:  {documents[0][:150]}")
print(f"Tokenized: {tokenized_train[0][:150]}")

Pre-tokenizing 110,000 train docs (4 workers)...


### 5b. TF-IDF Architecture B + LR

In [ ]:
t0 = time.time()
vectorizer_b = TfidfVectorizer(max_features=10000)
X_train_b = vectorizer_b.fit_transform(tokenized_train)
print(f"TF-IDF B: {X_train_b.shape} | Time: {time.time() - t0:.1f}s")
print(f"Sample features: {list(vectorizer_b.get_feature_names_out()[2500:2520])}")

In [ ]:
t0 = time.time()
lr_model_b = LinearRegression()
lr_model_b.fit(X_train_b, prices)
print(f"LR B training: {time.time() - t0:.1f}s")

def lr_tfidf_arch_b(item):
    tokenized = tokenized_test_map[item.summary]
    x = vectorizer_b.transform([tokenized])
    return max(lr_model_b.predict(x)[0], 0)

results_lr_b = evaluate(lr_tfidf_arch_b, test)

---
## Step 6: Ensemble Models (dung Architecture B)

Tat ca ensemble dung `X_train_b` + `tokenized_test_map` (da cache).

### 6a. Random Forest (subset 40K, 300 trees)

In [ ]:
np.random.seed(SEED)
SUBSET_RF = 40_000
t0 = time.time()
rf_model = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=N_JOBS)
rf_model.fit(X_train_b[:SUBSET_RF], prices[:SUBSET_RF])
print(f"RF training: {time.time() - t0:.1f}s (on {SUBSET_RF:,} samples)")

def random_forest_pricer(item):
    tokenized = tokenized_test_map[item.summary]
    x = vectorizer_b.transform([tokenized])
    return max(0, rf_model.predict(x)[0])

results_rf = evaluate(random_forest_pricer, test)

### 6b. XGBoost (full data, GPU)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
xgb_model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    random_state=SEED,
    n_jobs=N_JOBS,
    device="cuda",          # RTX 5060 Ti
    tree_method="hist",
)
xgb_model.fit(X_train_b, prices)
print(f"XGBoost training (GPU): {time.time() - t0:.1f}s")

def xgboost_pricer(item):
    tokenized = tokenized_test_map[item.summary]
    x = vectorizer_b.transform([tokenized])
    return max(0, xgb_model.predict(x)[0])

results_xgb = evaluate(xgboost_pricer, test)

### 6c. LightGBM (full data)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000, learning_rate=0.1, random_state=SEED,
    n_jobs=N_JOBS, verbose=-1,
)
lgb_model.fit(X_train_b, prices)
print(f"LightGBM training: {time.time() - t0:.1f}s")

def lightgbm_pricer(item):
    tokenized = tokenized_test_map[item.summary]
    x = vectorizer_b.transform([tokenized])
    return max(0, lgb_model.predict(x)[0])

results_lgb = evaluate(lightgbm_pricer, test)

### 6d. CatBoost (full data, GPU)

In [ ]:
np.random.seed(SEED)
t0 = time.time()
cb_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.1, random_seed=SEED,
    verbose=0, task_type="GPU",    # RTX 5060 Ti
)
cb_model.fit(X_train_b, prices)
print(f"CatBoost training (GPU): {time.time() - t0:.1f}s")

def catboost_pricer(item):
    tokenized = tokenized_test_map[item.summary]
    x = vectorizer_b.transform([tokenized])
    return max(0, cb_model.predict(x)[0])

results_cb = evaluate(catboost_pricer, test)

---
## Final Summary

In [ ]:
summary = pd.DataFrame([
    {"Model": "Random", **results_random},
    {"Model": "Mean", **results_mean},
    {"Model": "Median", **results_median},
    {"Model": "LR + TF-IDF (Arch A)", **results_lr_a},
    {"Model": "LR + TF-IDF (Arch B)", **results_lr_b},
    {"Model": "Random Forest (Arch B)", **results_rf},
    {"Model": "XGBoost GPU (Arch B)", **results_xgb},
    {"Model": "LightGBM (Arch B)", **results_lgb},
    {"Model": "CatBoost GPU (Arch B)", **results_cb},
])
summary = summary.set_index("Model")
summary.style.format({
    "rmsle": "{:.4f}",
    "mae": "{:,.0f}",
    "mape": "{:.1f}",
    "r2": "{:.1f}",
}).highlight_min(subset=["rmsle", "mae", "mape"], color="lightgreen").highlight_max(subset=["r2"], color="lightgreen")